# 1. Setup

Runs locally. Open this notebook from inside the repo (the setup cell finds the repo root automatically) and install the packages once with the venv's Python:

```
python -m pip install -r requirements.txt
```

Repo must have minimally `./common` and `./data/processed` folders. Trained models are saved to `./models/` and metrics to `./experiments/training/`.

In [30]:
import os, sys
from pathlib import Path

# Repo root = first folder (cwd or a parent) that contains ./common and ./data.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "common").is_dir() and (p / "data").is_dir())
os.chdir(ROOT)
sys.path.append(str(ROOT))

print("cwd:", os.getcwd())
print("root contents:", sorted(os.listdir('.')))
print("processed:", sorted(os.listdir('data/processed')))

cwd: c:\NUS\Y3S1\DSA2111\project\ai-detection-with-cefr
root contents: ['.git', '.gitignore', '.vscode', 'README.md', 'common', 'data', 'experiments', 'models', 'requirements.txt', 'training']
processed: ['test.parquet', 'test_heldout_generator.parquet', 'train.parquet', 'val.parquet']


In [31]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 3080


# 2. Training and evaluation code

The code lives in `training/` so that calibration, the backend and the experiments can import it:

| File | Contents |
|---|---|
| `config.py` | paths, `TrainConfig`, every run's settings (`CONFIGS`), `FINAL_MODEL` |
| `data.py` | loading the parquet splits, tokenizing, class weights |
| `model_factory.py` | the base model with its new 8-class head, optionally wrapped with LoRA |
| `metrics.py` | `compute_metrics` for training; CEFR scores and confusion matrices |
| `train.py` | `train`, `run_all`, ordinal training targets, checkpoint clean-up |
| `inference.py` | `predict_probs(texts, ckpt)`: (n, 8) probabilities, used by every other module |
| `evaluate.py` | evaluation on test and the held-out generator set; the original 2-class model as the baseline |
| `results_log.py`, `plot_sweeps.py` | logging to `experiments/training/results1.xlsx`; the sweep figures |

To add an experiment, add its runs to `CONFIGS` in `config.py` under new names, then add a section below.

In [ ]:
import json

from training.config import CONFIGS, EXP_DIR, FINAL_MODEL, MODELS_DIR, RUNS_DIR
from training.evaluate import eval_original_binary_model, evaluate, seed_summary
from training.inference import predict_probs
from training.results_log import log_experiment
from training.train import drop_checkpoints, run_all, train


def train_and_evaluate(names) -> None:
    """Train each config, delete its checkpoint folders, then evaluate it."""
    run_all(names)
    drop_checkpoints(names)
    evaluate(names)

# 3. Experiments

Each section changes one thing from the previous one and says why. Models are saved to `models/<name>/`,
results to `experiments/training/runs/`, and every experiment is logged to `experiments/training/results1.xlsx`.

**How the best setting is chosen (decided before Exp 3):** highest validation macro-F1. Gaps under 0.01 count as
ties and are broken by validation CEFR within-one-level accuracy. AI recall, held-out recall and the highest
false-flag rate must not get noticeably worse. Test results are reported, never used to choose.

## Smoke test

One epoch on 200 texts, to check the pipeline after code changes. Its scores are meaningless.

In [38]:
train(CONFIGS.get('smoke'), limit=200, overwrite=True)

[train] config=smoke  cuda=True  limit=200


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at ahmediqbal/ai-text-detector-model and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[train] trainable params: 817,928


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

[train] class weights: [2.7780001163482666, 1.6670000553131104, 1.0, 0.7350000143051147, 0.6579999923706055, 0.7580000162124634, 1.7860000133514404, 0.781000018119812]


Epoch,Training Loss,Validation Loss,Macro F1,Auroc Ai Vs Human,F1 Native,F1 A1,F1 A2,F1 B1,F1 B2,F1 C1,F1 C2,F1 Ai
1,No log,2.045229,0.118673,0.776875,0.000000,0.000000,0.320611,0.000000,0.000000,0.120000,0.000000,0.508772


[train] smoke: {'config': 'smoke', 'trainable_params': 817928, 'train_time_s': 2.2, 'val_macro_f1': 0.11867282710593277, 'val_ai_auroc': 0.7768752986144289, 'test_macro_f1': 0.18703616352201258, 'test_ai_auroc': 0.6784519827998089, 'platform': 'Windows-11-10.0.26200-SP0'}


{'config': 'smoke',
 'trainable_params': 817928,
 'train_time_s': 2.2,
 'val_macro_f1': 0.11867282710593277,
 'val_ai_auroc': 0.7768752986144289,
 'test_macro_f1': 0.18703616352201258,
 'test_ai_auroc': 0.6784519827998089,
 'platform': 'Windows-11-10.0.26200-SP0'}

## Exp 1: Colab runs (3 epochs, LoRA lr 1e-4)

The first runs, trained on Google Colab (Tesla T4): full fine-tuning and LoRA at ranks 4, 8 and 16. The models
were copied to `models/e1_*`, and their training metrics (validation macro-F1, training time) recorded from the
Colab output into `experiments/training/e1_*_metrics.json`. Here they are re-evaluated with the current metrics,
together with the original 2-class model as the reference.

**Result:** LoRA far behind (validation macro-F1 0.36 vs 0.43; AI recall 55% vs 98%). LoRA's best epoch was its
last, so it was still improving.

In [ ]:
for split in ("test", "test_heldout_generator"):
    eval_original_binary_model(split)

EXP1 = ["e1_full_ft", "e1_lora_r4", "e1_lora_r8", "e1_lora_r16"]
evaluate(EXP1)
log_experiment("Reference: original model", ["baseline"])
log_experiment("Exp 1: Colab, 3 ep, LoRA lr 1e-4", EXP1)

## Exp 2: more epochs (LoRA 10, full fine-tuning 4)

Exp 1's models were still improving at their last epoch, so LoRA gets 10 epochs and full fine-tuning 4.

**Result:** LoRA catches up on validation macro-F1 (0.45-0.46 vs 0.45) but still detects less AI. Rank (4, 8, 16)
makes no difference, as in Exp 1.

In [ ]:
EXP2 = ["e2_full_ft", "e2_lora_r4", "e2_lora_r8", "e2_lora_r16"]
train_and_evaluate(EXP2)
log_experiment("Exp 2: more epochs (LoRA 10, full FT 4)", EXP2)

## Exp 3: learning-rate sweeps

LoRA still lagged at lr 1e-4. Since rank made no difference, LoRA is fixed at r = 8 and its learning rate swept
over its usual range; full fine-tuning gets a sweep too, so both methods are compared at their best. The
lr 1e-4 (LoRA) and 2e-5 (full fine-tuning) points are the Exp 2 models.

**Result:** LoRA peaks at 5e-4 (0.508), full fine-tuning at 1e-4 (0.489) and collapses at 5e-4 (0.350).

In [ ]:
EXP3_NEW = [f"e3_lora_r8_lr{t}" for t in ("5e-5", "3e-4", "5e-4", "1e-3")] + \
           [f"e3_full_ft_lr{t}" for t in ("1e-5", "5e-5", "1e-4", "3e-4", "5e-4")]
train_and_evaluate(EXP3_NEW)

EXP3 = ["e3_lora_r8_lr5e-5", "e2_lora_r8", "e3_lora_r8_lr3e-4", "e3_lora_r8_lr5e-4", "e3_lora_r8_lr1e-3",
        "e3_full_ft_lr1e-5", "e2_full_ft", "e3_full_ft_lr5e-5", "e3_full_ft_lr1e-4",
        "e3_full_ft_lr3e-4", "e3_full_ft_lr5e-4"]
log_experiment("Exp 3: learning-rate sweep (LoRA r=8, full FT)", EXP3)

## Exp 4: ordinal smoothing (LoRA r = 8, lr 5e-4)

Ordered-level training targets (see `ordinal_targets`) spread some probability to the neighbouring CEFR levels,
so a one-level mistake costs less. Smoothing 0 is the Exp 3 winner.

**Result:** flat (validation macro-F1 0.496-0.514, within noise). By the selection rule, 0.3 and 0 tie on
macro-F1 and 0 wins on within-one-level accuracy, so **no smoothing** is kept.

In [ ]:
EXP4_NEW = ["e4_lora_r8_lr5e-4_s01", "e4_lora_r8_lr5e-4_s02", "e4_lora_r8_lr5e-4_s03"]
train_and_evaluate(EXP4_NEW)

SMOOTHING_RUNS = {0.0: "e3_lora_r8_lr5e-4", 0.1: "e4_lora_r8_lr5e-4_s01", 0.2: "e4_lora_r8_lr5e-4_s02", 0.3: "e4_lora_r8_lr5e-4_s03"}
(RUNS_DIR / "smoothing_sweep_runs.json").write_text(json.dumps({str(k): v for k, v in SMOOTHING_RUNS.items()}, indent=2))
log_experiment("Exp 4: ordinal smoothing sweep (LoRA r=8, lr 5e-4)", list(SMOOTHING_RUNS.values()))

## Final: chosen settings, 3 seeds each

The chosen LoRA (r = 8, lr 5e-4, no smoothing) and full fine-tuning (lr 1e-4, 4 epochs) are repeated with
seeds 1 and 2 (the original runs used seed 2111), to measure how much identical runs vary.

**Result:** identical runs vary by about ±0.014 validation macro-F1, so the two methods are tied on validation
(0.499 vs 0.503). LoRA catches more AI (test 99.8% vs 97.0%, held-out 96.8% vs 93.6%); full fine-tuning is better
at CEFR (test macro-F1 0.551 vs 0.530) and flags fewer humans.

In [ ]:
FINAL_LORA = ["e3_lora_r8_lr5e-4", "final_lora_seed1", "final_lora_seed2"]
FINAL_FFT = ["e3_full_ft_lr1e-4", "final_fft_seed1", "final_fft_seed2"]
train_and_evaluate(["final_lora_seed1", "final_lora_seed2", "final_fft_seed1", "final_fft_seed2"])
log_experiment("Final: chosen settings, 3 seeds each", FINAL_LORA + FINAL_FFT)

print("LoRA r=8, lr 5e-4:", seed_summary(FINAL_LORA))
print("Full FT, lr 1e-4: ", seed_summary(FINAL_FFT))

## Figures

Learning-rate sweep (Exp 3) and smoothing sweep (Exp 4), saved to `experiments/training/figures/`.
Close the images in other programs first; Windows won't let an open file be overwritten.

In [ ]:
from IPython.display import Image, display

from training import plot_sweeps

plot_sweeps.FIG_DIR.mkdir(parents=True, exist_ok=True)
display(Image(plot_sweeps.lr_sweep()))
display(Image(plot_sweeps.smoothing_sweep(SMOOTHING_RUNS)))